In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count

def main():
    # Create Spark session with local master and file system configuration
    spark = (
        SparkSession.builder
        .appName("read_gold_dim_customers")
        .master("local[*]")
        .config("spark.hadoop.fs.defaultFS", "file:///")
        .getOrCreate()
    )
    
    # Define the path to the gold dimension table
    gold_path = "file:/data/gold/dim_customers"
    
    # Print header information
    print("Reading Gold dimension: dim_customers")
    print(f"Path: {gold_path}")
    
    # Read the parquet files from gold layer
    df = spark.read.parquet(gold_path)
    
    # Display the schema of the dimension table
    print("\nSchema:")
    df.printSchema()
    
    # Show sample data from all records
    print("\nSample data:")
    df.show(truncate=False)
    
    # Filter and display only current records (is_current = true)
    print("\nCurrent records only (is_current = true):")
    df_current = df.filter(col("is_current") == True)
    df_current.show(truncate=False)
    
    # Count versions per customer to identify customers with multiple versions
    print("\nVersions per customer:")
    (
        df.groupBy("customer_code")
        .agg(count("*").alias("versions"))
        .orderBy(col("versions").desc())
        .show()
    )
    
    # Create a temporary view for SQL queries
    print("\nCreating temp view: dim_customers")
    df.createOrReplaceTempView("dim_customers")
    
    # Query current dimension records using SQL
    print("\nQuerying current dimension records:")
    spark.sql("""
        SELECT
            customer_sk,
            customer_code,
            customer_name,
            customer_address,
            customer_phone,
            customer_email,
            birth_date,
            valid_from,
            valid_to
        FROM dim_customers
        WHERE is_current = true
        ORDER BY customer_code
    """).show(truncate=False)
    
    # Optional: Show historical changes for customers with multiple versions
    print("\nCustomers with historical changes:")
    spark.sql("""
        SELECT
            customer_code,
            COUNT(*) as version_count
        FROM dim_customers
        GROUP BY customer_code
        HAVING COUNT(*) > 1
        ORDER BY version_count DESC
    """).show()
    
    # Optional: Show full history of a specific customer (example with customer_code = 1)
    print("\nFull history example (if customer has versions):")
    spark.sql("""
        SELECT
            customer_sk,
            customer_code,
            customer_name,
            customer_address,
            customer_phone,
            customer_email,
            birth_date,
            valid_from,
            valid_to,
            is_current
        FROM dim_customers
        WHERE customer_code = 1
        ORDER BY valid_from
    """).show(truncate=False)
    
    # Stop the Spark session
    spark.stop()

if __name__ == "__main__":
    main()

Reading Gold dimension: dim_customers
Path: file:/data/gold/dim_customers

Schema:
root
 |-- customer_code: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_address: string (nullable = true)
 |-- customer_phone: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- birth_date: timestamp (nullable = true)
 |-- customer_sk: integer (nullable = true)
 |-- valid_from: date (nullable = true)
 |-- valid_to: date (nullable = true)
 |-- is_current: boolean (nullable = true)


Sample data:
+-------------+-------------------+--------------------------------+--------------+--------------------------+-------------------+-----------+----------+--------+----------+
|customer_code|customer_name      |customer_address                |customer_phone|customer_email            |birth_date         |customer_sk|valid_from|valid_to|is_current|
+-------------+-------------------+--------------------------------+--------------+-------------------------